# Graded Response Model — EQSQ (Single Scale)

Fits a single-dimensional GRM to all 120 EQSQ items.

In [1]:
%load_ext autoreload
%autoreload 2

import os, sys
os.environ['JAX_PLATFORMS'] = 'cpu'
sys.path.insert(0, os.path.dirname(os.getcwd()))

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from plot_helpers import (plot_loss_comparison, plot_forest_discriminations,
                          plot_ability_scatter, plot_ability_distributions,
                          plot_thresholds, plot_individual_abilities,
                          plot_imputation_weights_pcolormesh)

## 1. Load Data

In [2]:
from bayesianquilts.data.eqsq import get_data, item_keys, response_cardinality

df, num_people = get_data(polars_out=True)
print(f"Dataset: {num_people} people, {len(item_keys)} items, {response_cardinality} categories")
df.head()

Dataset: 13256 people, 120 items, 4 categories


person,E1,E2,E3,E4,E5,E6,E7,E8,E9,E10,E11,E12,E13,E14,E15,E16,E17,E18,E19,E20,E21,E22,E23,E24,E25,E26,E27,E28,E29,E30,E31,E32,E33,E34,E35,E36,…,S24,S25,S26,S27,S28,S29,S30,S31,S32,S33,S34,S35,S36,S37,S38,S39,S40,S41,S42,S43,S44,S45,S46,S47,S48,S49,S50,S51,S52,S53,S54,S55,S56,S57,S58,S59,S60
u32,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,…,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
0,2,0,2,1,3,2,0,2,0,2,0,0,3,3,3,2,2,0,1,3,2,3,0,0,2,1,2,2,3,2,1,0,3,3,1,1,…,0,2,1,2,2,3,2,1,0,3,3,1,1,3,3,3,0,1,3,0,2,1,3,1,2,3,1,0,3,1,3,2,1,0,3,3,2
1,3,2,2,1,1,1,1,2,3,3,2,3,1,0,0,2,1,3,2,0,2,3,2,3,0,3,3,2,2,0,3,1,0,0,1,3,…,3,0,3,3,2,2,0,3,1,0,0,1,3,0,1,1,3,1,3,1,2,3,2,2,1,0,2,2,2,3,1,2,3,3,2,3,1
2,2,2,2,0,0,1,1,1,2,0,3,1,1,1,2,2,2,1,0,0,0,2,3,1,0,3,1,3,0,2,1,1,0,0,1,1,…,1,0,3,1,3,0,2,1,1,0,0,1,1,0,3,1,3,0,2,3,3,3,2,3,0,0,2,3,2,0,2,0,2,3,2,1,3
3,1,1,1,0,1,0,2,1,3,3,1,2,3,2,1,2,1,1,1,3,1,0,0,0,2,0,1,2,0,0,1,3,1,2,0,2,…,0,2,0,1,2,0,0,1,3,1,2,0,2,1,0,1,1,1,2,1,1,2,2,2,2,3,1,2,1,2,1,2,1,2,2,3,0
4,2,0,1,3,3,1,2,2,1,2,0,0,3,0,3,2,0,0,2,3,1,3,0,1,2,0,2,0,2,3,1,0,3,3,1,2,…,1,2,0,2,0,2,3,1,0,3,3,1,2,2,2,3,1,2,1,0,3,2,3,0,2,2,3,1,0,3,0,2,1,0,2,3,0


In [3]:
SUBSAMPLE_N = num_people
sub_df = df
print(f"Using full dataset: N = {SUBSAMPLE_N}")

Using full dataset: N = 13256


## 2. Prepare Data

In [4]:
def make_data_dict(dataframe):
    data = {}
    for col in dataframe.columns:
        arr = dataframe[col].to_numpy().astype(np.float64)
        data[col] = arr
    data['person'] = np.arange(len(dataframe), dtype=np.float64)
    return data

batch = make_data_dict(sub_df)
n_bad = sum(np.sum(np.isnan(batch[k]) | (batch[k] < 0) | (batch[k] >= response_cardinality)) for k in item_keys)
print(f"Bad/missing values: {n_bad}")

BATCH_SIZE = 512
steps_per_epoch = int(np.ceil(SUBSAMPLE_N / BATCH_SIZE))
print(f"N: {SUBSAMPLE_N}, Batch size: {BATCH_SIZE}, Steps per epoch: {steps_per_epoch}")

def data_factory():
    indices = np.arange(SUBSAMPLE_N)
    np.random.shuffle(indices)
    for start in range(0, SUBSAMPLE_N, BATCH_SIZE):
        end = min(start + BATCH_SIZE, SUBSAMPLE_N)
        idx_batch = indices[start:end]
        yield {k: v[idx_batch] for k, v in batch.items()}

Bad/missing values: 7420
N: 13256, Batch size: 512, Steps per epoch: 26


## 3. Fit Baseline GRM (Ignorable Missingness)

In [5]:
from bayesianquilts.irt.grm import GRModel

model_baseline = GRModel(
    item_keys=item_keys,
    num_people=SUBSAMPLE_N,
    dim=1,
    response_cardinality=response_cardinality,
    dtype=jnp.float64,
)

NUM_EPOCHS = 200
SNAPSHOT_EPOCH = 50

res_baseline = model_baseline.fit(
    data_factory,
    batch_size=BATCH_SIZE,
    dataset_size=SUBSAMPLE_N,
    num_epochs=NUM_EPOCHS,
    steps_per_epoch=steps_per_epoch,
    learning_rate=1e-4,
    patience=10,
    clip_norm=1.,
    zero_nan_grads=True,
    snapshot_epoch=SNAPSHOT_EPOCH,
    lr_decay_factor=0.975,
)
losses_baseline = res_baseline[0]
snapshot_params = res_baseline[2] if len(res_baseline) > 2 else None
print(f"Baseline final loss: {losses_baseline[-1]:.2f}")
if snapshot_params is not None:
    print(f"Snapshot saved at epoch {SNAPSHOT_EPOCH}")

--- Starting Training ---
Patience for early stopping: 10 epochs
LR decay factor on plateau: 0.975
Convergence will be checked every: 1 epoch(s)
Checkpoints will be saved to: None
Optimizing keys: ['mu\\identity\\normal\\loc', 'mu\\identity\\normal\\scale', 'difficulties0\\identity\\normal\\loc', 'difficulties0\\identity\\normal\\scale', 'discriminations\\softplus\\normal\\loc', 'discriminations\\softplus\\normal\\scale', 'eta\\softplus\\normal\\loc', 'eta\\softplus\\normal\\scale', 'kappa\\softplus\\igamma\\concentration', 'kappa\\softplus\\igamma\\scale', 'kappa_a\\softplus\\igamma\\concentration', 'kappa_a\\softplus\\igamma\\scale', 'ddifficulties\\softplus\\normal\\loc', 'ddifficulties\\softplus\\normal\\scale', 'abilities\\identity\\normal\\loc', 'abilities\\identity\\normal\\scale']
-------------------------


  -> New best loss found. Checkpoint saved.                    


  -> New best loss found. Checkpoint saved.                    


  -> New best loss found. Checkpoint saved.                    


Epoch 4/200 (LR: 0.000100):  58%|█████▊    | 15/26 [00:00<00:00, 47.39batch/s, best_loss=242052.0362, loss=239722.6602]

...

  -> New best loss found. Checkpoint saved.                    


  -> New best loss found. Checkpoint saved.                    


  -> New best loss found. Checkpoint saved.                    


  -> New best loss found. Checkpoint saved.                    


Epoch 155/200 (LR: 0.000005):  58%|█████▊    | 15/26 [00:00<00:00, 40.65batch/s, best_loss=100574.4179, loss=100732.4636]

In [ ]:
model_baseline.save_to_disk('grm_baseline')

In [ ]:
# Calibrate baseline model early so we can compute WAIC for mixed imputation
def calibrate_manually(model, n_samples=32, seed=42):
    try:
        surrogate = model.surrogate_distribution_generator(model.params)
        key = jax.random.PRNGKey(seed)
        samples = surrogate.sample(n_samples, seed=key)
        expectations = {k: jnp.mean(v, axis=0) for k, v in samples.items()}
        model.calibrated_expectations = expectations
        model.surrogate_sample = samples
    except KeyError as e:
        print(f"  Warning: surrogate sampling failed ({e}), using point estimates")
        point_estimates = {}
        for key_name, value in model.params.items():
            parts = key_name.split('\\')
            if len(parts) >= 4:
                param_name = parts[0]
                if parts[-2] == 'normal' and parts[-1] == 'loc':
                    point_estimates[param_name] = value
        model.calibrated_expectations = point_estimates

calibrate_manually(model_baseline, n_samples=32, seed=101)

## 4. Fit MICEBayesianLOO Imputation Model

In [ ]:
from bayesianquilts.imputation.mice_loo import MICEBayesianLOO

imputation_df = sub_df.select(item_keys).to_pandas()
imputation_df = imputation_df.replace(-1, float('nan'))
print(f"NaN count: {imputation_df.isna().sum().sum()}")

mice_loo = MICEBayesianLOO(
    random_state=42,
    prior_scale=1.0,
    pathfinder_num_samples=100,
    pathfinder_maxiter=50,
    batch_size=512,
    verbose=True,
)
mice_loo.fit_loo_models(
    X_df=imputation_df,
    fit_zero_predictors=True,
    n_jobs=1,
    n_top_features=60,
    seed=42,
)

NaN count: 7420
Computing feature correlations...
Fitting MICE Bayesian LOO-CV models with Pathfinder
  Variables: 120
  Observations: 13256
  Min obs per model: 5
  Parallel jobs: 1
  Top features per target: 60
  Global Ordinal Values: [0. 1. 2. 3.] (n=4)

Fitting zero-predictor models...
  Scheduling 120 zero-predictor jobs...
  Var 0 (E1): n_obs=13220, elpd/n=-0.0520
  Var 1 (E2): n_obs=13136, elpd/n=-0.0427
  Var 2 (E3): n_obs=13229, elpd/n=-0.0461
  Var 3 (E4): n_obs=13140, elpd/n=-0.0506
  Var 4 (E5): n_obs=13199, elpd/n=-0.0530
  Var 5 (E6): n_obs=13200, elpd/n=-0.0529
  Var 6 (E7): n_obs=13212, elpd/n=-0.0482
  Var 7 (E8): n_obs=13222, elpd/n=-0.0506
  Var 8 (E9): n_obs=13218, elpd/n=-0.0499


In [ ]:
mice_loo.save('mice_loo_model.yaml')

In [ ]:
from bayesianquilts.imputation.mixed import IrtMixedImputationModel

mixed_imputation = IrtMixedImputationModel(
    irt_model=model_baseline,
    mice_model=mice_loo,
    data_factory=data_factory,
    irt_elpd_batch_size=4,
)

print(mixed_imputation.summary())

## 5. Fit GRM with Analytic Rao-Blackwellized Imputation

In [ ]:
model_imputed = GRModel(
    item_keys=item_keys,
    num_people=SUBSAMPLE_N,
    dim=1,
    response_cardinality=response_cardinality,
    dtype=jnp.float64,
    imputation_model=mixed_imputation,
)

if snapshot_params is not None:
    print(f"Warm-starting from baseline epoch-{SNAPSHOT_EPOCH} snapshot")

res_imputed = model_imputed.fit(
    data_factory,
    batch_size=BATCH_SIZE,
    dataset_size=SUBSAMPLE_N,
    num_epochs=NUM_EPOCHS,
    steps_per_epoch=steps_per_epoch,
    learning_rate=1e-4,
    clip_norm=1.,
    patience=10,
    zero_nan_grads=True,
    initial_values=snapshot_params,
    lr_decay_factor=0.975,
)
losses_imputed = res_imputed[0]
print(f"Imputed final loss: {losses_imputed[-1]:.2f}")

In [ ]:
model_imputed.save_to_disk('grm_imputed')

## 6. Compare Results

In [ ]:
fig = plot_loss_comparison(losses_baseline, losses_imputed, title='EQSQ')
fig.savefig('loss_comparison.pdf', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# calibrate_manually already defined above; just calibrate the imputed model
calibrate_manually(model_imputed, n_samples=32, seed=102)

In [ ]:
fig = plot_forest_discriminations(item_keys, model_baseline, model_imputed,
                                  title='EQSQ \u2014 Item Discriminations')
fig.savefig('discriminations.pdf', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
ab_base = np.array(model_baseline.calibrated_expectations['abilities']).flatten()
ab_imp = np.array(model_imputed.calibrated_expectations['abilities']).flatten()

fig = plot_ability_scatter(ab_base, ab_imp, label='emotional quotient')
fig.savefig('ability_scatter.pdf', bbox_inches='tight', dpi=150)
plt.show()

fig = plot_ability_distributions(ab_base, ab_imp, label='emotional quotient')
fig.savefig('ability_distributions.pdf', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
fig = plot_thresholds(item_keys, model_baseline, model_imputed,
                      title='EQSQ \u2014 Difficulty Thresholds')
fig.savefig('thresholds.pdf', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
fig = plot_individual_abilities(item_keys, model_baseline, model_imputed)
fig.savefig('individual_abilities.pdf', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
fig = plot_imputation_weights_pcolormesh(mice_loo, mixed_imputation, item_keys,
                                         title='EQSQ \u2014 Imputation Weights')
fig.savefig('imputation_weights.pdf', bbox_inches='tight', dpi=150)
plt.show()

## Summary

This notebook fitted a single-scale Graded Response Model to all 120 EQSQ
items (E1–E60 + S1–S60) with 4 response categories. Two models were compared:

1. **Baseline** — ignorable missingness (zero-fill for missing values)
2. **Imputed** — analytic Rao-Blackwellized imputation via MICEBayesianLOO

The discrimination parameters indicate how well each item differentiates
respondents along the single latent dimension. The ability distributions
show the estimated latent trait values under each approach.